In [11]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, GlobalAveragePooling2D, GlobalMaxPooling2D, Reshape, Dense, multiply, Add, Lambda, Concatenate

def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    shared_layer_one = Dense(channel//ratio, activation='relu', kernel_initializer='he_normal', use_bias=True, bias_initializer='zeros')
    shared_layer_two = Dense(channel, kernel_initializer='he_normal', use_bias=True, bias_initializer='zeros')

    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1,1,channel))(avg_pool)
    avg_pool = shared_layer_one(avg_pool)
    avg_pool = shared_layer_two(avg_pool)

    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1,1,channel))(max_pool)
    max_pool = shared_layer_one(max_pool)
    max_pool = shared_layer_two(max_pool)

    cbam_feature = Add()([avg_pool, max_pool])
    cbam_feature = Activation('sigmoid')(cbam_feature)

    return multiply([input_feature, cbam_feature])

def spatial_attention(input_feature):
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(input_feature)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(input_feature)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    cbam_feature = Conv2D(filters=1, kernel_size=(7,7), padding='same', activation='sigmoid', kernel_initializer='he_normal', use_bias=False)(concat)

    return multiply([input_feature, cbam_feature])

def cbam_block(cbam_feature, ratio=8):
    cbam_feature = channel_attention(cbam_feature, ratio)
    cbam_feature = spatial_attention(cbam_feature)
    return cbam_feature


In [3]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, Conv2D, Activation, Add, Multiply, Lambda, Reshape, GlobalMaxPooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, LearningRateScheduler
from tensorflow.keras.regularizers import l2
from tensorflow.keras.mixed_precision import set_global_policy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [4]:
# Enable mixed precision training
set_global_policy('mixed_float16')

In [5]:
# Define paths
train_csv_path = '/Users/sanchitthakur/Desktop/real_major/PA-100K/train.csv'
val_csv_path = '/Users/sanchitthakur/Desktop/real_major/PA-100K/val.csv'
test_csv_path = '/Users/sanchitthakur/Desktop/real_major/PA-100K/test.csv'
image_dir = '/Users/sanchitthakur/Desktop/real_major/PA-100K/data'

In [6]:
# Load annotations
train_annotations = pd.read_csv(train_csv_path)
val_annotations = pd.read_csv(val_csv_path)
test_annotations = pd.read_csv(test_csv_path)

# Define image size and batch size
IMG_SIZE = (224, 224)
BATCH_SIZE = 16  # Reduce batch size

In [7]:
# Create data generators with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.5, 1.5],
    zoom_range=0.2,
    shear_range=0.2
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

def create_generator(datagen, annotations, image_dir):
    generator = datagen.flow_from_dataframe(
        dataframe=annotations,
        directory=image_dir,
        x_col='Image',  # Correct column name for image IDs
        y_col=annotations.columns[1:],
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='raw'
    )
    return generator

train_generator = create_generator(train_datagen, train_annotations, image_dir)
val_generator = create_generator(val_test_datagen, val_annotations, image_dir)
test_generator = create_generator(val_test_datagen, test_annotations, image_dir)


Found 80000 validated image filenames.
Found 10000 validated image filenames.
Found 10000 validated image filenames.


In [12]:
# Load pre-trained MobileNetV3 model
base_model = MobileNetV3Small(input_shape=(224, 224, 3), include_top=False, weights='imagenet')


In [13]:
# Add CBAM attention module
x = base_model.output
x = cbam_block(x)
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu', kernel_regularizer=l2(0.01))(x)
x = BatchNormalization()(x)
x = Dropout(0.6)(x)
predictions = Dense(26, activation='sigmoid', kernel_regularizer=l2(0.01))(x)  # PA-100K has 26 attributes


In [14]:
# Create the model
model = Model(inputs=base_model.input, outputs=predictions)


In [15]:
# Freeze the base model layers
for layer in base_model.layers:
    layer.trainable = False

In [16]:
# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [17]:
# Callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-7)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

def scheduler(epoch, lr):
    if epoch < 10:
        return lr
    else:
        return lr * tf.math.exp(-0.1)

lr_scheduler = LearningRateScheduler(scheduler)


In [18]:
# Train the model
history = model.fit(train_generator, epochs=10, steps_per_epoch=len(train_generator), validation_data=val_generator, validation_steps=len(val_generator), callbacks=[reduce_lr, early_stopping, lr_scheduler])

/opt/homebrew/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
 193/5000 ━━━━━━━━━━━━━━━━━━━━ 7:36 95ms/step - accuracy: 0.3280 - loss: 2.8136

KeyboardInterrupt: 

In [ ]:
# Fine-tune the model
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])

history_fine = model.fit(train_generator, epochs=10, steps_per_epoch=len(train_generator), validation_data=val_generator, validation_steps=len(val_generator), callbacks=[reduce_lr, early_stopping, lr_scheduler])


In [ ]:
# Plot training & validation accuracy and loss curves
def plot_training_curves(history):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Model accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')

    plt.show()

plot_training_curves(history)
plot_training_curves(history_fine)

In [ ]:
# Generate confusion matrices and classification report for validation and test data
def generate_confusion_matrices(generator, model, num_attributes):
    generator.reset()
    predictions = model.predict(generator, steps=len(generator))
    predictions = (predictions > 0.5).astype(int)  # Threshold predictions

    true_labels = []
    for i in range(len(generator)):
        true_labels.append(generator[i][1])
    true_labels = np.array(true_labels)

    for i in range(num_attributes):
        cm = confusion_matrix(true_labels[:, i], predictions[:, i])
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(cmap=plt.cm.Blues)
        plt.title(f'Confusion Matrix for Attribute {i}')
        plt.show()

    report = classification_report(true_labels, predictions, target_names=[f'Attribute {i}' for i in range(num_attributes)])
    print(report)

num_attributes = 26

print("Validation Data:")
generate_confusion_matrices(val_generator, model, num_attributes)

print("Test Data:")
generate_confusion_matrices(test_generator, model, num_attributes)

In [ ]:
# Function to predict attributes from a single image
def predict_image(image_path, model):
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)
    predictions = (predictions > 0.5).astype(int)  # Threshold predictions

    attribute_labels = [f'Attribute {i}' for i in range(num_attributes)]
    predicted_labels = [attribute_labels[i] for i in range(len(predictions[0])) if predictions[0][i] == 1]

    print("Predicted Attributes:")
    for label in predicted_labels:
        print(label)

# Example usage
image_path = 'path/to/your/image.jpg'
predict_image(image_path, model)